In [2]:
# load code 15 csv file

import pandas as pd

exams = pd.read_csv('/media/Volume/data/CODE15/nkclean_360_12l/exams_labelled.csv')
exams.head()

,Unnamed: 0.1,Unnamed: 0,exam_id,age,is_male,nn_predicted_age,1dAVb,RBBB,LBBB,SB,ST,AF,patient_id,death,timey,normal_ecg,trace_file,r_peaks,r_peak_interval_mean,r_peak_variance
0,0,0,1169160,38,True,40.160484,False,False,False,False,False,False,523632,False,2.098628,True,exams_part13.hdf5,NaN,NaN,NaN
1,1,1,2873686,73,True,67.059440,False,False,False,False,False,False,1724173,False,6.657529,False,exams_part13.hdf5,NaN,NaN,NaN
2,2,2,168405,67,True,79.621740,False,False,False,False,False,True,51421,False,4.282188,False,exams_part13.hdf5,NaN,NaN,NaN
3,3,3,271011,41,True,69.750260,False,False,False,False,False,False,1737282,False,4.038353,True,exams_part13.hdf5,NaN,NaN,NaN
4,4,4,384368,73,True,78.873460,False,False,False,False,False,False,331652,False,3.786298,False,exams_part13.hdf5,NaN,NaN,NaN


In [8]:
from dataset.generic_utils import find_records

records = find_records('/media/Volume/data/CODE15/nkclean_360_12l/')
print(len(records))

Finding records in /media/Volume/data/CODE15/nkclean_360_12l/...
344699


In [3]:
import os
import wfdb
import wfdb.processing as wp

from pandarallel import pandarallel
pandarallel.initialize(progress_bar=True)

def analyze_ecg(row):
    """
    Analyze the ecg signal

    :param row: the row of the dataframe
    :return: the peaks
    """
    record_path = os.path.join('/media/Volume/data/CODE15/nkclean_360_12l', str(row['exam_id']), str(row['exam_id']))
    # check path
    if not os.path.exists(record_path + '.dat'):
        print(f'Path {record_path} does not exist')
        return None
    
    record = wfdb.rdrecord(record_path)
    peaks = wp.xqrs_detect(record.p_signal[:, 1], record.fs, verbose=False)
    return peaks

exams['r_peaks'] = exams.parallel_apply(analyze_ecg, axis=1)

INFO: Pandarallel will run on 16 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.


Path /media/Volume/data/CODE15/nkclean_360_12l/1047867/1047867 does not exist
Path /media/Volume/data/CODE15/nkclean_360_12l/1108322/1108322 does not exist
Path /media/Volume/data/CODE15/nkclean_360_12l/4411244/4411244 does not exist
Path /media/Volume/data/CODE15/nkclean_360_12l/1165739/1165739 does not exist
Path /media/Volume/data/CODE15/nkclean_360_12l/332035/332035 does not exist
Path /media/Volume/data/CODE15/nkclean_360_12l/457852/457852 does not exist
Path /media/Volume/data/CODE15/nkclean_360_12l/4266642/4266642 does not exist
Path /media/Volume/data/CODE15/nkclean_360_12l/102019/102019 does not exist
Path /media/Volume/data/CODE15/nkclean_360_12l/1931119/1931119 does not exist


Process ForkPoolWorker-17:
Process ForkPoolWorker-16:
Process ForkPoolWorker-10:
Process ForkPoolWorker-15:
Process ForkPoolWorker-11:
Process ForkPoolWorker-12:
Process ForkPoolWorker-9:
Traceback (most recent call last):
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
Process ForkPoolWorker-4:
  File "/usr/local/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
Traceback (most recent call last):
Process ForkPoolWorker-14:
Traceback (most recent call last):
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/multiprocessing/pool.py", line 125, in worker
    result = (True, func(*args, **kwd

KeyboardInterrupt: 

Process ForkPoolWorker-27:
Process ForkPoolWorker-29:
Process ForkPoolWorker-25:
Traceback (most recent call last):
Process ForkPoolWorker-23:
Process ForkPoolWorker-20:
Traceback (most recent call last):
Process ForkPoolWorker-18:
  File "/usr/local/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
Process ForkPoolWorker-22:
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/usr/local/lib/python3.11/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
Process ForkPoolWorker-24:
Process ForkPoolWorker-26:
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/usr/local/lib/python3.11/multiprocessing/pool.py", line 114, in worker
    task = get()
           ^^^^^
  File "/usr/local/lib/python3.11/multiprocessing/process.py", line 108

In [7]:
import numpy as np

exams['r_peak_interval_mean'] = exams.parallel_apply(lambda row: np.mean((row['r_peaks'][1:] - row['r_peaks'][:-1]) / 360) if row['r_peaks'] is not None else None, axis=1)
exams['r_peak_variance'] = exams.parallel_apply(lambda row: np.std((row['r_peaks'][1:] - row['r_peaks'][:-1]) / 360) if row['r_peaks'] is not None else None, axis=1)
exams.head()

/home/luna97/Research/MIT-BIH_ecg_arrhytmia/mit_bih_env/lib/python3.11/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/luna97/Research/MIT-BIH_ecg_arrhytmia/mit_bih_env/lib/python3.11/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/luna97/Research/MIT-BIH_ecg_arrhytmia/mit_bih_env/lib/python3.11/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/luna97/Research/MIT-BIH_ecg_arrhytmia/mit_bih_env/lib/python3.11/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/luna97/Research/MIT-BIH_ecg_arrhytmia/mit_bih_env/lib/python3.11/site-packages/numpy/_core/_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  r

/home/luna97/Research/MIT-BIH_ecg_arrhytmia/mit_bih_env/lib/python3.11/site-packages/numpy/_core/_methods.py:223: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/luna97/Research/MIT-BIH_ecg_arrhytmia/mit_bih_env/lib/python3.11/site-packages/numpy/_core/_methods.py:223: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/luna97/Research/MIT-BIH_ecg_arrhytmia/mit_bih_env/lib/python3.11/site-packages/numpy/_core/_methods.py:223: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/luna97/Research/MIT-BIH_ecg_arrhytmia/mit_bih_env/lib/python3.11/site-packages/numpy/_core/_methods.py:223: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/luna97/Research/MIT-BIH_ecg_arrhytmia/mit_bih_env/lib/python3.11/site-packages/numpy/_core/_method

,exam_id,age,is_male,nn_predicted_age,1dAVb,RBBB,LBBB,SB,ST,AF,patient_id,death,timey,normal_ecg,trace_file,r_peaks,r_peak_interval_mean,r_peak_variance
0,1169160,38,True,40.160484,False,False,False,False,False,False,523632,False,2.098628,True,exams_part13.hdf5,"[57, 285, 507, 742, 972, 1196, 1419, 1630, 184...",0.625309,0.020557
1,2873686,73,True,67.059440,False,False,False,False,False,False,1724173,False,6.657529,False,exams_part13.hdf5,"[219, 460, 702, 944, 1176, 1404, 1631, 1859, 2...",0.646181,0.020969
2,168405,67,True,79.621740,False,False,False,False,False,True,51421,False,4.282188,False,exams_part13.hdf5,"[220, 532, 716, 1024, 1200, 1599, 1874]",0.765741,0.214907
3,271011,41,True,69.750260,False,False,False,False,False,False,1737282,False,4.038353,True,exams_part13.hdf5,"[306, 585, 868, 1142, 1434, 1725, 2004, 2290, ...",0.795679,0.024149
4,384368,73,True,78.873460,False,False,False,False,False,False,331652,False,3.786298,False,exams_part13.hdf5,"[115, 285, 454, 623, 793, 962, 1132, 1303, 147...",0.473090,0.002729


In [8]:
exams.to_csv('/media/Volume/data/CODE15/exams.csv')